In [1]:
!nvidia-smi

Wed Oct  7 01:50:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/28.2 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 22.2 MB/s eta 0:00:00


In [1]:
import os
import json

# Create project structure
os.makedirs("data", exist_ok=True)
print("✅ Project structure ready")

✅ Project structure ready


In [2]:
from google.colab import files
uploaded = files.upload()  # Select preference_pairs.jsonl

for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
    print(f"✅ Uploaded: {filename}")

KeyboardInterrupt: 

In [3]:
sample_pairs = [
    {
        "prompt": "Movie: The Conjuring\nOverview: Paranormal investigators help a family terrorized by a dark presence.\nGenres: Horror, Mystery, Thriller\nRating: 7.5\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Not safe for children\nExplanation: This movie is in the Horror genre and contains frightening supernatural imagery that is not appropriate for children.",
        "rejected": "Classification: Safe for children\nExplanation: The movie has a high rating."
    },
    {
        "prompt": "Movie: Finding Nemo\nOverview: A clownfish sets out on a journey to find his son.\nGenres: Animation, Adventure, Comedy\nRating: 8.2\n\nIs this movie appropriate for children aged 5-10?",
        "chosen": "Classification: Safe for children\nExplanation: This is an animated, family-friendly film with positive messages about family and friendship.",
        "rejected": "Classification: Not safe for children\nExplanation: The movie has some intense scenes."
    },
]

with open("data/preference_pairs.jsonl", "w") as f:
    for pair in sample_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"✅ Created {len(sample_pairs)} sample preference pairs")

✅ Created 2 sample preference pairs


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="data/preference_pairs.jsonl",
    split="train"
)

print(f"✅ Loaded {len(dataset)} preference pairs")
print(f"Columns: {dataset.column_names}")

Generating train split: 0 examples [00:00, ? examples/s]

✅ Loaded 2 preference pairs
Columns: ['prompt', 'chosen', 'rejected']


In [5]:
from unsloth import FastLanguageModel, PatchDPOTrainer

PatchDPOTrainer()  # ✅ Required for Unsloth DPO

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/zephyr-sft-bnb-4bit",  # Or your SFT model
    max_seq_length=1024,
    load_in_4bit=True,
)

print("✅ Model loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.1: Fast Mistral patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/zephyr-sft-bnb-4bit as a legacy tokenizer.


✅ Model loaded


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ LoRA adapters added")

Unsloth 2026.10.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapters added


In [7]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=3,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs_dpo",
        report_to="none",
    ),
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=1024,
    max_prompt_length=512,
)

print("✅ DPO Trainer configured")

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
Unsloth: `max_prompt_length` is not supported by the installed TRL's DPOConfig and will be IGNORED - filter overlong prompts out of your dataset before training instead.


Adding EOS to train dataset (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s…

✅ DPO Trainer configured


In [8]:
print("🚀 Starting DPO training...")
dpo_trainer.train()
print("✅ DPO training complete!")

🚀 Starting DPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 167,772,160 of 7,577,276,416 (2.21% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logits / chosen,logits / rejected,logps / chosen,logps / rejected
1,0.693147,0.000000,0.000000,0.000000,0.000000,-3.171138,-3.016323,-83.885284,-78.945175
2,0.693147,0.000000,0.000000,0.000000,0.000000,-3.171138,-3.016323,-83.885284,-78.945175
3,0.018921,2.451773,-1.565125,1.000000,4.016898,-3.067602,-2.953156,-59.367558,-94.596420


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-3/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs_dpo/checkpoint-3.


✅ DPO training complete!


In [9]:
from google.colab import drive
drive.mount('/content/drive')

output_path = "/content/drive/MyDrive/movie-content-safety/dpo_aligned_model"
os.makedirs(output_path, exist_ok=True)

dpo_trainer.model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

print(f"✅ Model saved to Google Drive: {output_path}")

Mounted at /content/drive


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/movie-content-safety/dpo_aligned_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/movie-content-safety/dpo_aligned_model.


✅ Model saved to Google Drive: /content/drive/MyDrive/movie-content-safety/dpo_aligned_model


In [10]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_prompt = """Movie: The Conjuring
Overview: Paranormal investigators help a family terrorized by a dark presence.
Genres: Horror, Mystery, Thriller
Rating: 7.5

Is this movie appropriate for children aged 5-10?"""

messages = [
    {"role": "system", "content": "You are a movie content safety classifier."},
    {"role": "user", "content": test_prompt},
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([input_text], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.1,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"\n📌 DPO Model Response:\n{response.strip()}")

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📌 DPO Model Response:
No, this movie is not appropriate for children aged 5-10. The Conjuring is a horror movie with intense and frightening supernatural elements that are not suitable for young children. It may cause nightmares and fear in children who are not mature enough to handle such content. It is recommended for viewers aged 16 and above.
